# KLA PS1 — Final Training Pipeline

## Final submitted model: NAFNetSR + High-Frequency Loss

This notebook reproduces the training workflow used for the submitted model.

**Final architecture**
- NAFNet-style encoder/decoder restoration network
- 2× super-resolution through the final PixelShuffle reconstruction block
- Bicubic ×2 global reconstruction baseline
- Residual prediction added to the bicubic baseline

**Final training objective**
- L1 reconstruction loss
- High-Frequency L1 loss
- `HF_WEIGHT = 0.05`

**Training data**
- Official paired GT / NoisyLR data
- Training samples use the calibrated synthetic degradation generator
- Validation uses the real paired NoisyLR images

The notebook keeps the original student-workflow analysis and calibrated synthetic degradation pipeline from the submitted development notebook, followed by the final NAFNetSR training stage.


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

# --------------------------------------------------
# Device
# --------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch :", torch.__version__)
print("Device  :", DEVICE)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

# --------------------------------------------------
# Dataset paths
# --------------------------------------------------

DATASET_ROOT = Path("/kaggle/input/datasets/bhoomisaraf/semicon26")

TRAIN_GT_DIR = DATASET_ROOT / "train" / "train" / "GT"
TRAIN_LR_DIR = DATASET_ROOT / "train" / "train" / "NoisyLR"
TEST_LR_DIR  = DATASET_ROOT / "Test_NoisyLR" / "NoisyLR"

print("\nPaths:")
print("GT     :", TRAIN_GT_DIR)
print("NoisyLR:", TRAIN_LR_DIR)
print("Test   :", TEST_LR_DIR)

In [ ]:
for name, path in {
    "TRAIN_GT": TRAIN_GT_DIR,
    "TRAIN_LR": TRAIN_LR_DIR,
    "TEST_LR": TEST_LR_DIR,
}.items():

    print(f"{name}:")
    print(f"  exists = {path.exists()}")
    print(f"  path   = {path}")
    print()

In [ ]:
gt_files = sorted(TRAIN_GT_DIR.glob("*.npy"))
lr_files = sorted(TRAIN_LR_DIR.glob("*.npy"))
test_files = sorted(TEST_LR_DIR.glob("*.npy"))

print("Training GT     :", len(gt_files))
print("Training NoisyLR:", len(lr_files))
print("Test NoisyLR    :", len(test_files))

print("\nFirst 10 GT files:")
for p in gt_files[:10]:
    print(p.name)

print("\nFirst 10 NoisyLR files:")
for p in lr_files[:10]:
    print(p.name)

print("\nFirst 10 Test files:")
for p in test_files[:10]:
    print(p.name)

In [ ]:
gt_names = {p.name for p in gt_files}
lr_names = {p.name for p in lr_files}

print("GT files       :", len(gt_names))
print("LR files       :", len(lr_names))
print("Matched names  :", len(gt_names & lr_names))
print("GT only        :", len(gt_names - lr_names))
print("LR only        :", len(lr_names - gt_names))

In [ ]:
print("\nGT-only examples:")
for x in sorted(gt_names - lr_names)[:10]:
    print(x)

print("\nLR-only examples:")
for x in sorted(lr_names - gt_names)[:10]:
    print(x)
    

In [ ]:
# Create filename-based lookup
lr_lookup = {p.name: p for p in lr_files}

# Build paired dataset
pairs = [
    (gt_path, lr_lookup[gt_path.name])
    for gt_path in gt_files
    if gt_path.name in lr_lookup
]

# Safety checks
assert len(pairs) == len(gt_files), "Some GT files do not have matching LR files."
assert len(pairs) == len(lr_files), "Some LR files do not have matching GT files."

print("Total verified pairs:", len(pairs))

print("\nFirst 5 pairs:")
for gt_path, lr_path in pairs[:5]:
    print(f"GT: {gt_path.name}")
    print(f"LR: {lr_path.name}")
    print()

In [ ]:
# Load one complete training pair

gt_sample = np.load(pairs[0][0])
lr_sample = np.load(pairs[0][1])

print("GT")
print(" shape :", gt_sample.shape)
print(" dtype :", gt_sample.dtype)
print(" min   :", gt_sample.min())
print(" max   :", gt_sample.max())
print(" mean  :", gt_sample.mean())
print(" std   :", gt_sample.std())

print("\nNoisyLR")
print(" shape :", lr_sample.shape)
print(" dtype :", lr_sample.dtype)
print(" min   :", lr_sample.min())
print(" max   :", lr_sample.max())
print(" mean  :", lr_sample.mean())
print(" std   :", lr_sample.std())

In [ ]:
from collections import Counter

gt_shapes = Counter()
lr_shapes = Counter()
scale_factors = Counter()

bad_pairs = []

for gt_path, lr_path in pairs:
    gt = np.load(gt_path, mmap_mode="r")
    lr = np.load(lr_path, mmap_mode="r")

    gt_shapes[gt.shape] += 1
    lr_shapes[lr.shape] += 1

    if gt.ndim != 2 or lr.ndim != 2:
        bad_pairs.append((gt_path.name, gt.shape, lr.shape))
        continue

    scale_y = gt.shape[0] / lr.shape[0]
    scale_x = gt.shape[1] / lr.shape[1]

    scale_factors[(scale_y, scale_x)] += 1


print("GT shapes:")
for shape, count in gt_shapes.items():
    print(f"  {shape}: {count}")

print("\nNoisyLR shapes:")
for shape, count in lr_shapes.items():
    print(f"  {shape}: {count}")

print("\nScale factors:")
for scale, count in scale_factors.items():
    print(f"  {scale}: {count}")

print("\nBad/non-2D pairs:", len(bad_pairs))

In [ ]:
def audit_ranges(files, label):

    global_min = float("inf")
    global_max = float("-inf")

    total_pixels = 0
    below_zero = 0
    above_one = 0

    means = []
    stds = []

    for path in files:
        x = np.load(path, mmap_mode="r")

        global_min = min(global_min, float(x.min()))
        global_max = max(global_max, float(x.max()))

        total_pixels += x.size
        below_zero += int(np.sum(x < 0))
        above_one += int(np.sum(x > 1))

        means.append(float(x.mean()))
        stds.append(float(x.std()))

    print(f"===== {label} =====")
    print(f"Global min       : {global_min:.6f}")
    print(f"Global max       : {global_max:.6f}")
    print(f"Mean image mean  : {np.mean(means):.6f}")
    print(f"Mean image std   : {np.mean(stds):.6f}")
    print(f"Pixels < 0       : {below_zero:,}")
    print(f"Pixels > 1       : {above_one:,}")
    print(f"% below 0        : {100 * below_zero / total_pixels:.6f}%")
    print(f"% above 1        : {100 * above_one / total_pixels:.6f}%")
    print()


audit_ranges(gt_files, "GT")
audit_ranges(lr_files, "TRAIN NoisyLR")

In [ ]:
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(
    pairs,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

print("Total pairs :", len(pairs))
print("Train pairs :", len(train_pairs))
print("Val pairs   :", len(val_pairs))

In [ ]:
train_names = {gt.name for gt, _ in train_pairs}
val_names = {gt.name for gt, _ in val_pairs}

overlap = train_names & val_names

print("Train/Val overlap:", len(overlap))

assert len(overlap) == 0, "DATA LEAKAGE DETECTED"

print("✓ No train/validation filename overlap.")

In [ ]:
class KLARestorationDataset(Dataset):

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        gt_path, lr_path = self.pairs[idx]

        # Load exact floating-point arrays
        gt = np.load(gt_path).astype(np.float32)
        lr = np.load(lr_path).astype(np.float32)

        # Expected format:
        # GT: [H, W]
        # LR: [h, w]

        if gt.ndim != 2:
            raise ValueError(
                f"Expected 2D GT, got {gt.shape} for {gt_path.name}"
            )

        if lr.ndim != 2:
            raise ValueError(
                f"Expected 2D LR, got {lr.shape} for {lr_path.name}"
            )

        # [H,W] -> [1,H,W]
        gt = torch.from_numpy(gt).unsqueeze(0)
        lr = torch.from_numpy(lr).unsqueeze(0)

        return {
            "lr": lr,
            "gt": gt,
            "name": gt_path.name
        }

In [ ]:
train_dataset = KLARestorationDataset(train_pairs)
val_dataset = KLARestorationDataset(val_pairs)

print("Training samples  :", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

In [ ]:
batch = next(iter(train_loader))

lr = batch["lr"]
gt = batch["gt"]

print("LR batch")
print(" shape :", lr.shape)
print(" dtype :", lr.dtype)
print(" min   :", lr.min().item())
print(" max   :", lr.max().item())
print(" mean  :", lr.mean().item())

print("\nGT batch")
print(" shape :", gt.shape)
print(" dtype :", gt.dtype)
print(" min   :", gt.min().item())
print(" max   :", gt.max().item())
print(" mean  :", gt.mean().item())

print("\nNames:")
print(batch["name"][:5])

In [ ]:
lr = batch["lr"][0, 0].numpy()
gt = batch["gt"][0, 0].numpy()

# Bicubic LR only for visual comparison
lr_up = F.interpolate(
    torch.from_numpy(lr)[None, None],
    size=gt.shape,
    mode="bicubic",
    align_corners=False
)[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(lr, cmap="gray")
axes[0].set_title("Original NoisyLR 128×128")
axes[0].axis("off")

axes[1].imshow(lr_up, cmap="gray")
axes[1].set_title("Bicubic NoisyLR 256×256")
axes[1].axis("off")

axes[2].imshow(gt, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("GT 256×256")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from skimage.metrics import peak_signal_noise_ratio
from skimage.metrics import structural_similarity

def calculate_psnr(pred, target):
    """
    pred, target: numpy arrays in [0,1]
    """
    return peak_signal_noise_ratio(
        target,
        pred,
        data_range=1.0
    )


def calculate_ssim(pred, target):
    """
    pred, target: numpy arrays in [0,1]
    """
    return structural_similarity(
        target,
        pred,
        data_range=1.0
    )

Degradation Analysis

In [ ]:
def downsample_gt(gt, mode):

    gt_tensor = torch.from_numpy(
        gt.astype(np.float32)
    )[None, None]

    if mode == "area":

        out = F.interpolate(
            gt_tensor,
            size=(128, 128),
            mode="area"
        )

    elif mode == "bilinear":

        out = F.interpolate(
            gt_tensor,
            size=(128, 128),
            mode="bilinear",
            align_corners=False
        )

    elif mode == "bicubic":

        out = F.interpolate(
            gt_tensor,
            size=(128, 128),
            mode="bicubic",
            align_corners=False
        )

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return out[0, 0].numpy()

In [ ]:
def analyze_downsampling_residual(
    pairs,
    mode,
    max_samples=None
):

    selected_pairs = pairs

    if max_samples is not None:
        selected_pairs = pairs[:max_samples]

    residual_mean = []
    residual_std = []
    residual_abs_mean = []

    for gt_path, lr_path in selected_pairs:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        gt_down = downsample_gt(
            gt,
            mode
        )

        residual = lr - gt_down

        residual_mean.append(
            residual.mean()
        )

        residual_std.append(
            residual.std()
        )

        residual_abs_mean.append(
            np.abs(residual).mean()
        )

    return {
        "mode": mode,
        "mean_residual": np.mean(residual_mean),
        "std_residual": np.mean(residual_std),
        "mean_abs_residual": np.mean(residual_abs_mean),
    }

In [ ]:
downsampling_analysis = []

for mode in [
    "area",
    "bilinear",
    "bicubic"
]:

    result = analyze_downsampling_residual(
        train_pairs,
        mode,
        max_samples=1000
    )

    downsampling_analysis.append(result)

downsampling_df = pd.DataFrame(
    downsampling_analysis
)

display(downsampling_df)

In [ ]:
gt_path, lr_path = train_pairs[0]

gt = np.load(
    gt_path
).astype(np.float32)

lr = np.load(
    lr_path
).astype(np.float32)

gt_down = downsample_gt(
    gt,
    "area"
)

residual = lr - gt_down

print("Residual statistics")
print("=" * 40)
print("Mean :", residual.mean())
print("Std  :", residual.std())
print("Min  :", residual.min())
print("Max  :", residual.max())

plt.figure(figsize=(8, 5))

plt.hist(
    residual.flatten(),
    bins=200
)

plt.xlabel("LR − Downsampled GT")
plt.ylabel("Frequency")
plt.title("Observed Degradation Residual")

plt.show()

In [ ]:
def speckle_dependency_analysis(
    pairs,
    mode="area",
    max_samples=1000
):

    signal_values = []
    residual_values = []

    selected_pairs = pairs[:max_samples]

    for gt_path, lr_path in selected_pairs:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        gt_down = downsample_gt(
            gt,
            mode
        )

        residual = lr - gt_down

        signal_values.append(
            gt_down.reshape(-1)
        )

        residual_values.append(
            residual.reshape(-1)
        )

    signal = np.concatenate(
        signal_values
    )

    residual = np.concatenate(
        residual_values
    )

    return signal, residual

In [ ]:
signal, residual = speckle_dependency_analysis(
    train_pairs,
    mode="area",
    max_samples=1000
)

In [ ]:
from scipy.stats import pearsonr

abs_residual = np.abs(residual)

correlation, p_value = pearsonr(
    signal,
    abs_residual
)

print(
    f"Correlation between signal intensity "
    f"and |residual|: {correlation:.6f}"
)

print(
    f"p-value: {p_value:.6e}"
)

In [ ]:
bins = np.linspace(
    0,
    1,
    21
)

bin_centers = []
bin_std = []

for low, high in zip(
    bins[:-1],
    bins[1:]
):

    mask = (
        (signal >= low) &
        (signal < high)
    )

    if mask.sum() > 100:

        bin_centers.append(
            (low + high) / 2
        )

        bin_std.append(
            np.std(
                residual[mask]
            )
        )

plt.figure(figsize=(8, 5))

plt.plot(
    bin_centers,
    bin_std,
    marker="o"
)

plt.xlabel("Downsampled GT intensity")
plt.ylabel("Residual standard deviation")
plt.title("Noise Magnitude vs Signal Intensity")

plt.grid(True)

plt.show()

In [ ]:
safe_signal = np.maximum(
    signal,
    0.05
)

relative_residual = (
    residual / safe_signal
)

print("Relative residual statistics")
print("=" * 40)

print(
    "Mean:",
    np.mean(relative_residual)
)

print(
    "Std :",
    np.std(relative_residual)
)

print(
    "Median:",
    np.median(relative_residual)
)

print(
    "P1:",
    np.percentile(relative_residual, 1)
)

print(
    "P99:",
    np.percentile(relative_residual, 99)
)

In [ ]:
# --------------------------------------------------
# Spatial residual visualization
# --------------------------------------------------

# Select one representative validation/training pair
gt_path, lr_path = train_pairs[0]

# Load original arrays
gt_single = np.load(
    gt_path
).astype(np.float32)

lr_single = np.load(
    lr_path
).astype(np.float32)

# Downsample GT to LR resolution
gt_down_single = downsample_gt(
    gt_single,
    "area"
)

# Calculate 2D residual
residual_2d = lr_single - gt_down_single

print("Shapes:")
print("GT              :", gt_single.shape)
print("Downsampled GT  :", gt_down_single.shape)
print("NoisyLR         :", lr_single.shape)
print("Residual        :", residual_2d.shape)

print("\nResidual statistics:")
print("Mean :", residual_2d.mean())
print("Std  :", residual_2d.std())
print("Min  :", residual_2d.min())
print("Max  :", residual_2d.max())


# --------------------------------------------------
# Visualization
# --------------------------------------------------

fig, axes = plt.subplots(
    1,
    4,
    figsize=(18, 5)
)

axes[0].imshow(
    gt_down_single,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[0].set_title("Downsampled GT")
axes[0].axis("off")

axes[1].imshow(
    lr_single,
    cmap="gray"
)
axes[1].set_title("Observed NoisyLR")
axes[1].axis("off")

# Use symmetric limits so positive/negative residuals
# are visually comparable
res_limit = np.percentile(
    np.abs(residual_2d),
    99
)

axes[2].imshow(
    residual_2d,
    cmap="gray",
    vmin=-res_limit,
    vmax=res_limit
)
axes[2].set_title("Residual")
axes[2].axis("off")

axes[3].imshow(
    np.abs(residual_2d),
    cmap="magma"
)
axes[3].set_title("|Residual|")
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.optimize import curve_fit

# --------------------------------------------------
# Recompute signal/residual using the same diagnostic
# --------------------------------------------------

signal, residual_flat = speckle_dependency_analysis(
    train_pairs,
    mode="bicubic",
    max_samples=1000
)

# Avoid very dark pixels where relative estimates become unstable
mask = signal >= 0.02

signal_fit = signal[mask]
residual_fit = residual_flat[mask]

# --------------------------------------------------
# Bin by signal intensity
# --------------------------------------------------

bins = np.linspace(0.02, 1.0, 41)

bin_centers = []
observed_std = []

for low, high in zip(bins[:-1], bins[1:]):

    m = (
        (signal_fit >= low) &
        (signal_fit < high)
    )

    if np.sum(m) > 500:

        bin_centers.append(
            (low + high) / 2
        )

        observed_std.append(
            np.std(residual_fit[m])
        )

bin_centers = np.asarray(bin_centers)
observed_std = np.asarray(observed_std)

# --------------------------------------------------
# Model:
#
# sigma(x) = sqrt(sigma_g^2 + (sigma_s*x)^2)
# --------------------------------------------------

def noise_std_model(x, sigma_g, sigma_s):

    return np.sqrt(
        sigma_g**2 +
        (sigma_s * x)**2
    )

# Initial guess
initial_guess = [
    0.01,   # additive Gaussian
    0.15    # multiplicative/speckle
]

params, covariance = curve_fit(
    noise_std_model,
    bin_centers,
    observed_std,
    p0=initial_guess,
    bounds=(
        [0.0, 0.0],
        [1.0, 2.0]
    )
)

sigma_g_est, sigma_s_est = params

print("Estimated heteroscedastic noise model")
print("=" * 50)
print(f"Estimated additive sigma     : {sigma_g_est:.6f}")
print(f"Estimated multiplicative sigma: {sigma_s_est:.6f}")

In [ ]:
fitted_std = noise_std_model(
    bin_centers,
    sigma_g_est,
    sigma_s_est
)

plt.figure(figsize=(8, 5))

plt.plot(
    bin_centers,
    observed_std,
    marker="o",
    label="Observed residual std"
)

plt.plot(
    bin_centers,
    fitted_std,
    linewidth=2,
    label="Gaussian + multiplicative model"
)

plt.xlabel("Downsampled GT intensity")
plt.ylabel("Residual standard deviation")
plt.title("Observed vs Fitted Noise Model")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(
    observed_std,
    fitted_std
)

rmse = np.sqrt(
    np.mean(
        (observed_std - fitted_std) ** 2
    )
)

print("Fit quality")
print("=" * 40)
print(f"R²   : {r2:.6f}")
print(f"RMSE : {rmse:.6f}")

In [ ]:
# Additive-only:
# sigma(x) = sigma_g

def additive_only(x, sigma_g):
    return np.full_like(x, sigma_g)


# Multiplicative-only:
# sigma(x) = sigma_s * x

def multiplicative_only(x, sigma_s):
    return sigma_s * x


# Fit additive-only
params_add, _ = curve_fit(
    additive_only,
    bin_centers,
    observed_std,
    p0=[0.05],
    bounds=([0.0], [1.0])
)

# Fit multiplicative-only
params_mult, _ = curve_fit(
    multiplicative_only,
    bin_centers,
    observed_std,
    p0=[0.15],
    bounds=([0.0], [2.0])
)

pred_add = additive_only(
    bin_centers,
    params_add[0]
)

pred_mult = multiplicative_only(
    bin_centers,
    params_mult[0]
)

r2_add = r2_score(
    observed_std,
    pred_add
)

r2_mult = r2_score(
    observed_std,
    pred_mult
)

r2_combined = r2

print("Model comparison")
print("=" * 50)

print(
    f"Additive only       R²: {r2_add:.6f}"
)

print(
    f"Multiplicative only R²: {r2_mult:.6f}"
)

print(
    f"Combined            R²: {r2_combined:.6f}"
)

Build the synthetic degradation engine

In [ ]:
def add_gaussian_noise(
    image,
    sigma
):
    """
    Additive Gaussian noise.

    image : numpy array, float32
            expected clean image
    sigma : noise standard deviation
    """

    noise = np.random.normal(
        loc=0.0,
        scale=sigma,
        size=image.shape
    ).astype(np.float32)

    return image + noise

In [ ]:
def add_speckle_noise(
    image,
    sigma
):
    """
    Multiplicative speckle noise.

    y = x * (1 + n)

    n ~ N(0, sigma^2)
    """

    noise = np.random.normal(
        loc=0.0,
        scale=sigma,
        size=image.shape
    ).astype(np.float32)

    return image * (1.0 + noise)

In [ ]:
def downsample_2x(
    image,
    mode="bicubic"
):
    """
    Downsample 256×256 -> 128×128.
    """

    tensor = torch.from_numpy(
        image.astype(np.float32)
    )[None, None]

    if mode == "bicubic":

        output = F.interpolate(
            tensor,
            size=(128, 128),
            mode="bicubic",
            align_corners=False
        )

    elif mode == "bilinear":

        output = F.interpolate(
            tensor,
            size=(128, 128),
            mode="bilinear",
            align_corners=False
        )

    elif mode == "area":

        output = F.interpolate(
            tensor,
            size=(128, 128),
            mode="area"
        )

    else:
        raise ValueError(
            f"Unsupported downsampling mode: {mode}"
        )

    return output[0, 0].numpy()

In [ ]:
SIGMA_GAUSSIAN = 0.024346
SIGMA_SPECKLE = 0.167811


def synthetic_degrade(
    gt,
    sigma_gaussian=SIGMA_GAUSSIAN,
    sigma_speckle=SIGMA_SPECKLE,
    downsample_mode="bicubic"
):

    # 1. Downsample
    lr = downsample_2x(
        gt,
        mode=downsample_mode
    )

    # 2. Multiplicative / speckle noise
    lr = add_speckle_noise(
        lr,
        sigma=sigma_speckle
    )

    # 3. Additive Gaussian noise
    lr = add_gaussian_noise(
        lr,
        sigma=sigma_gaussian
    )

    return lr.astype(np.float32)

In [ ]:
gt_test = np.load(
    train_pairs[0][0]
).astype(np.float32)

real_lr_test = np.load(
    train_pairs[0][1]
).astype(np.float32)

synthetic_lr_test = synthetic_degrade(
    gt_test
)

print("GT:")
print(gt_test.shape, gt_test.min(), gt_test.max())

print("\nReal NoisyLR:")
print(
    real_lr_test.shape,
    real_lr_test.min(),
    real_lr_test.max(),
    real_lr_test.mean(),
    real_lr_test.std()
)

print("\nSynthetic NoisyLR:")
print(
    synthetic_lr_test.shape,
    synthetic_lr_test.min(),
    synthetic_lr_test.max(),
    synthetic_lr_test.mean(),
    synthetic_lr_test.std()
)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

axes[0].imshow(
    gt_test,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[0].set_title("GT")
axes[0].axis("off")

axes[1].imshow(
    real_lr_test,
    cmap="gray"
)
axes[1].set_title("Real NoisyLR")
axes[1].axis("off")

axes[2].imshow(
    synthetic_lr_test,
    cmap="gray"
)
axes[2].set_title("Synthetic NoisyLR")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def print_stats(name, x):

    print(f"\n{name}")
    print("=" * 40)

    print(f"Min  : {x.min():.6f}")
    print(f"Max  : {x.max():.6f}")
    print(f"Mean : {x.mean():.6f}")
    print(f"Std  : {x.std():.6f}")

    print(
        f"< 0  : {100 * np.mean(x < 0):.4f}%"
    )

    print(
        f"> 1  : {100 * np.mean(x > 1):.4f}%"
    )


print_stats(
    "Real NoisyLR",
    real_lr_test
)

print_stats(
    "Synthetic NoisyLR",
    synthetic_lr_test
)

In [ ]:
def collect_synthetic_statistics(
    pairs,
    max_samples=500
):

    real_stds = []
    synthetic_stds = []

    real_means = []
    synthetic_means = []

    real_below_zero = []
    synthetic_below_zero = []

    real_above_one = []
    synthetic_above_one = []

    selected_pairs = pairs[:max_samples]

    for gt_path, lr_path in selected_pairs:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        real_lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        synthetic_lr = synthetic_degrade(
            gt
        )

        real_means.append(
            real_lr.mean()
        )

        synthetic_means.append(
            synthetic_lr.mean()
        )

        real_stds.append(
            real_lr.std()
        )

        synthetic_stds.append(
            synthetic_lr.std()
        )

        real_below_zero.append(
            np.mean(real_lr < 0)
        )

        synthetic_below_zero.append(
            np.mean(synthetic_lr < 0)
        )

        real_above_one.append(
            np.mean(real_lr > 1)
        )

        synthetic_above_one.append(
            np.mean(synthetic_lr > 1)
        )

    return {
        "real_mean": np.mean(real_means),
        "synthetic_mean": np.mean(synthetic_means),

        "real_std": np.mean(real_stds),
        "synthetic_std": np.mean(synthetic_stds),

        "real_below_zero": np.mean(real_below_zero),
        "synthetic_below_zero": np.mean(synthetic_below_zero),

        "real_above_one": np.mean(real_above_one),
        "synthetic_above_one": np.mean(synthetic_above_one),
    }

In [ ]:
synthetic_stats = collect_synthetic_statistics(
    train_pairs,
    max_samples=500
)

for key, value in synthetic_stats.items():
    print(
        f"{key:25s}: {value:.6f}"
    )

In [ ]:
def compute_real_degradation_stats(
    pairs,
    mode="bicubic",
    max_samples=None
):

    if max_samples is None:
        selected_pairs = pairs
    else:
        selected_pairs = pairs[:max_samples]

    records = []

    for gt_path, lr_path in selected_pairs:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        gt_down = downsample_2x(
            gt,
            mode=mode
        )

        residual = lr - gt_down

        # Signal-dependent noise estimate
        signal = gt_down

        # Ignore extremely dark pixels
        mask = signal >= 0.02

        x = signal[mask]
        r = residual[mask]

        # Estimate multiplicative component
        # using r / x
        relative_r = r / np.maximum(x, 0.02)

        records.append({
            "mean": lr.mean(),
            "std": lr.std(),

            "residual_std": residual.std(),
            "residual_abs_mean": np.abs(residual).mean(),

            "below_zero": np.mean(lr < 0),
            "above_one": np.mean(lr > 1),

            "relative_std": relative_r.std(),

            "signal_mean": gt_down.mean()
        })

    return pd.DataFrame(records)

In [ ]:
real_deg_stats = compute_real_degradation_stats(
    train_pairs,
    mode="bicubic",
    max_samples=1000
)

display(
    real_deg_stats.describe()
)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

axes[0].hist(
    real_deg_stats["residual_std"],
    bins=40
)
axes[0].set_title("Real Residual Std")
axes[0].set_xlabel("Residual std")
axes[0].set_ylabel("Images")

axes[1].hist(
    real_deg_stats["relative_std"],
    bins=40
)
axes[1].set_title("Real Relative Noise Std")
axes[1].set_xlabel("Relative residual std")

axes[2].hist(
    real_deg_stats["above_one"],
    bins=40
)
axes[2].set_title("Real Fraction > 1")
axes[2].set_xlabel("Fraction > 1")

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
percentiles = [
    1,
    5,
    10,
    25,
    50,
    75,
    90,
    95,
    99
]

print("REAL DEGRADATION PERCENTILES")
print("=" * 60)

for column in [
    "residual_std",
    "relative_std",
    "above_one"
]:

    print(f"\n{column}")

    values = np.percentile(
        real_deg_stats[column],
        percentiles
    )

    for p, v in zip(
        percentiles,
        values
    ):

        print(
            f"P{p:02d}: {v:.6f}"
        )

Calibrated degradation strength

In [ ]:
# --------------------------------------------------
# Calibrated degradation statistics
# --------------------------------------------------

REAL_RESIDUAL_STD_P05 = 0.038306
REAL_RESIDUAL_STD_P50 = 0.083727
REAL_RESIDUAL_STD_P95 = 0.134768

BASELINE_RESIDUAL_STD = REAL_RESIDUAL_STD_P50

BASE_SIGMA_G = 0.024346
BASE_SIGMA_S = 0.167811

print("Degradation calibration")
print("=" * 50)

print(f"Base Gaussian sigma : {BASE_SIGMA_G:.6f}")
print(f"Base Speckle sigma  : {BASE_SIGMA_S:.6f}")
print(f"Median residual std : {BASELINE_RESIDUAL_STD:.6f}")

In [ ]:
# --------------------------------------------------
# Empirical severity distribution
# --------------------------------------------------

severity_values = real_deg_stats[
    "residual_std"
].to_numpy(
    dtype=np.float32
)

# Keep the central observed range.
# This avoids letting a few extreme outliers dominate
# synthetic training.
severity_low = np.percentile(
    severity_values,
    5
)

severity_high = np.percentile(
    severity_values,
    95
)

severity_values_clipped = severity_values[
    (severity_values >= severity_low) &
    (severity_values <= severity_high)
]

print("Severity distribution")
print("=" * 50)

print(f"P05 : {severity_low:.6f}")
print(f"P95 : {severity_high:.6f}")
print(f"Samples retained: {len(severity_values_clipped)}")

In [ ]:
def sample_degradation_strength():

    observed_residual_std = np.random.choice(
        severity_values_clipped
    )

    severity_scale = (
        observed_residual_std /
        BASELINE_RESIDUAL_STD
    )

    sigma_g = (
        BASE_SIGMA_G *
        severity_scale
    )

    sigma_s = (
        BASE_SIGMA_S *
        severity_scale
    )

    return (
        float(sigma_g),
        float(sigma_s),
        float(observed_residual_std),
        float(severity_scale)
    )

In [ ]:
for _ in range(10):

    sigma_g, sigma_s, observed_std, scale = (
        sample_degradation_strength()
    )

    print(
        f"target_std={observed_std:.4f} | "
        f"scale={scale:.3f} | "
        f"sigma_g={sigma_g:.4f} | "
        f"sigma_s={sigma_s:.4f}"
    )

In [ ]:
def synthetic_degrade_calibrated(
    gt,
    downsample_mode="bicubic"
):

    # --------------------------------------------------
    # Sample degradation strength from real data
    # --------------------------------------------------

    sigma_g, sigma_s, _, _ = (
        sample_degradation_strength()
    )

    # --------------------------------------------------
    # Candidate order:
    #
    # Downsample → Speckle → Gaussian
    #
    # This is our calibration order.
    # We will NOT yet claim this is the hidden order.
    # --------------------------------------------------

    lr = downsample_2x(
        gt,
        mode=downsample_mode
    )

    lr = add_speckle_noise(
        lr,
        sigma=sigma_s
    )

    lr = add_gaussian_noise(
        lr,
        sigma=sigma_g
    )

    return lr.astype(np.float32)

In [ ]:
synthetic_target_stds = []

for _ in range(5000):

    _, _, target_std, _ = (
        sample_degradation_strength()
    )

    synthetic_target_stds.append(
        target_std
    )

synthetic_target_stds = np.asarray(
    synthetic_target_stds
)

print("Synthetic sampled severity")
print("=" * 50)

for p in [5, 25, 50, 75, 95]:

    print(
        f"P{p:02d}: "
        f"{np.percentile(synthetic_target_stds, p):.6f}"
    )

In [ ]:
def collect_calibrated_synthetic_stats(
    pairs,
    max_samples=500
):

    records = []

    selected_pairs = pairs[:max_samples]

    for gt_path, lr_path in selected_pairs:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        real_lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        synthetic_lr = (
            synthetic_degrade_calibrated(gt)
        )

        records.append({

            "real_mean":
                real_lr.mean(),

            "synthetic_mean":
                synthetic_lr.mean(),

            "real_std":
                real_lr.std(),

            "synthetic_std":
                synthetic_lr.std(),

            "real_below_zero":
                np.mean(real_lr < 0),

            "synthetic_below_zero":
                np.mean(synthetic_lr < 0),

            "real_above_one":
                np.mean(real_lr > 1),

            "synthetic_above_one":
                np.mean(synthetic_lr > 1),
        })

    return pd.DataFrame(records)

In [ ]:
calibrated_stats = (
    collect_calibrated_synthetic_stats(
        train_pairs,
        max_samples=500
    )
)

display(
    calibrated_stats.mean().to_frame(
        "Mean across images"
    )
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

axes[0].hist(
    real_deg_stats["residual_std"],
    bins=40,
    alpha=0.7,
    label="Real"
)

axes[0].hist(
    severity_values_clipped,
    bins=40,
    alpha=0.7,
    label="Calibration range"
)

axes[0].set_title(
    "Observed Degradation Severity"
)

axes[0].set_xlabel(
    "Residual standard deviation"
)

axes[0].set_ylabel(
    "Images"
)

axes[0].legend()
axes[0].grid(True)


axes[1].hist(
    synthetic_target_stds,
    bins=40,
    alpha=0.7,
    label="Synthetic sampled"
)

axes[1].set_title(
    "Synthetic Severity Sampling"
)

axes[1].set_xlabel(
    "Target residual standard deviation"
)

axes[1].set_ylabel(
    "Samples"
)

axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

> Test the six possible degradation orders

In [ ]:
from itertools import permutations


def apply_degradation_operation(
    image,
    operation,
    sigma_g,
    sigma_s
):
    """
    Apply exactly one permitted degradation.
    """

    if operation == "downsample":

        return downsample_2x(
            image,
            mode="bicubic"
        )

    elif operation == "gaussian":

        return add_gaussian_noise(
            image,
            sigma=sigma_g
        )

    elif operation == "speckle":

        return add_speckle_noise(
            image,
            sigma=sigma_s
        )

    else:
        raise ValueError(
            f"Unknown operation: {operation}"
        )

In [ ]:
def synthetic_degrade_ordered(
    gt,
    order,
    sigma_g,
    sigma_s
):

    image = gt.astype(
        np.float32,
        copy=True
    )

    for operation in order:

        image = apply_degradation_operation(
            image,
            operation,
            sigma_g,
            sigma_s
        )

    return image.astype(
        np.float32
    )

In [ ]:
def apply_degradation_operation(
    image,
    operation,
    sigma_g,
    sigma_s
):
    """
    Apply exactly one permitted degradation:
    Gaussian noise, speckle noise, or 2x downsampling.
    """

    if operation == "downsample":

        return downsample_2x(
            image,
            mode="bicubic"
        )

    elif operation == "gaussian":

        return add_gaussian_noise(
            image,
            sigma=sigma_g
        )

    elif operation == "speckle":

        return add_speckle_noise(
            image,
            sigma=sigma_s
        )

    else:
        raise ValueError(
            f"Unknown degradation operation: {operation}"
        )

In [ ]:
def synthetic_degrade_ordered(
    gt,
    order,
    sigma_g,
    sigma_s
):
    """
    Apply the three permitted degradation operations
    in the specified order.
    """

    image = gt.astype(
        np.float32,
        copy=True
    )

    for operation in order:

        image = apply_degradation_operation(
            image=image,
            operation=operation,
            sigma_g=sigma_g,
            sigma_s=sigma_s
        )

    return image.astype(np.float32)

In [ ]:
print(synthetic_degrade_ordered)

In [ ]:
DEGRADATION_OPERATIONS = [
    "downsample",
    "gaussian",
    "speckle"
]

DEGRADATION_ORDERS = list(
    permutations(
        DEGRADATION_OPERATIONS
    )
)

for i, order in enumerate(
    DEGRADATION_ORDERS,
    start=1
):

    print(
        f"{i}. "
        + " → ".join(order)
    )

In [ ]:
gt_test = np.load(
    train_pairs[0][0]
).astype(np.float32)

real_lr_test = np.load(
    train_pairs[0][1]
).astype(np.float32)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 8)
)

# Real LR
axes[0, 0].imshow(
    real_lr_test,
    cmap="gray"
)

axes[0, 0].set_title(
    "Real NoisyLR"
)

axes[0, 0].axis("off")


for i, order in enumerate(
    DEGRADATION_ORDERS,
    start=1
):

    synthetic = synthetic_degrade_ordered(
        gt_test,
        order,
        sigma_g=BASE_SIGMA_G,
        sigma_s=BASE_SIGMA_S
    )

    ax = axes.flatten()[i]

    ax.imshow(
        synthetic,
        cmap="gray"
    )

    ax.set_title(
        " → ".join(order)
    )

    ax.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
def compare_degradation_order(
    pairs,
    order,
    sigma_g=BASE_SIGMA_G,
    sigma_s=BASE_SIGMA_S,
    max_samples=300
):

    records = []

    selected_pairs = pairs[:max_samples]

    for gt_path, lr_path in selected_pairs:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        real_lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        synthetic_lr = synthetic_degrade_ordered(
            gt,
            order,
            sigma_g,
            sigma_s
        )

        records.append({

            "real_mean":
                real_lr.mean(),

            "synthetic_mean":
                synthetic_lr.mean(),

            "real_std":
                real_lr.std(),

            "synthetic_std":
                synthetic_lr.std(),

            "real_below_zero":
                np.mean(real_lr < 0),

            "synthetic_below_zero":
                np.mean(synthetic_lr < 0),

            "real_above_one":
                np.mean(real_lr > 1),

            "synthetic_above_one":
                np.mean(synthetic_lr > 1),

            "real_min":
                real_lr.min(),

            "synthetic_min":
                synthetic_lr.min(),

            "real_max":
                real_lr.max(),

            "synthetic_max":
                synthetic_lr.max(),
        })

    return pd.DataFrame(records)

In [ ]:
order_results = []

for order in DEGRADATION_ORDERS:

    df_order = compare_degradation_order(
        train_pairs,
        order,
        sigma_g=BASE_SIGMA_G,
        sigma_s=BASE_SIGMA_S,
        max_samples=300
    )

    real_mean = df_order[
        "real_mean"
    ].mean()

    synthetic_mean = df_order[
        "synthetic_mean"
    ].mean()

    real_std = df_order[
        "real_std"
    ].mean()

    synthetic_std = df_order[
        "synthetic_std"
    ].mean()

    real_below = df_order[
        "real_below_zero"
    ].mean()

    synthetic_below = df_order[
        "synthetic_below_zero"
    ].mean()

    real_above = df_order[
        "real_above_one"
    ].mean()

    synthetic_above = df_order[
        "synthetic_above_one"
    ].mean()

    # Aggregate absolute distribution mismatch
    score = (
        abs(real_mean - synthetic_mean)
        +
        abs(real_std - synthetic_std)
        +
        abs(real_below - synthetic_below)
        +
        abs(real_above - synthetic_above)
    )

    order_results.append({

        "Order":
            " → ".join(order),

        "Real Mean":
            real_mean,

        "Synthetic Mean":
            synthetic_mean,

        "Real Std":
            real_std,

        "Synthetic Std":
            synthetic_std,

        "Real <0":
            real_below,

        "Synthetic <0":
            synthetic_below,

        "Real >1":
            real_above,

        "Synthetic >1":
            synthetic_above,

        "Mismatch Score":
            score
    })


order_results_df = pd.DataFrame(
    order_results
).sort_values(
    "Mismatch Score"
)

display(
    order_results_df
)

In [ ]:
# ============================================================
# FINAL CALIBRATED SYNTHETIC DEGRADATION
# ============================================================

FINAL_DEGRADATION_ORDER = (
    "gaussian",
    "downsample",
    "speckle"
)

def synthetic_degrade_final(gt):
    """
    Final calibrated synthetic degradation.

    Pipeline:
        GT
        -> Gaussian noise
        -> 2x downsampling
        -> Speckle noise

    Noise strengths are sampled from the
    empirically observed degradation severity.
    """

    sigma_g, sigma_s, target_std, scale = (
        sample_degradation_strength()
    )

    degraded = synthetic_degrade_ordered(
        gt=gt,
        order=FINAL_DEGRADATION_ORDER,
        sigma_g=sigma_g,
        sigma_s=sigma_s
    )

    return degraded.astype(np.float32), {
        "sigma_g": sigma_g,
        "sigma_s": sigma_s,
        "target_std": target_std,
        "scale": scale
    }

In [ ]:
gt_test = np.load(
    train_pairs[0][0]
).astype(np.float32)

real_lr_test = np.load(
    train_pairs[0][1]
).astype(np.float32)

synthetic_lr_test, degradation_info = (
    synthetic_degrade_final(gt_test)
)

print("Degradation parameters")
print("=" * 50)

for key, value in degradation_info.items():
    print(f"{key:12s}: {value:.6f}")

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

axes[0].imshow(
    gt_test,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[0].set_title("GT — 256×256")
axes[0].axis("off")

axes[1].imshow(
    real_lr_test,
    cmap="gray"
)
axes[1].set_title("Real NoisyLR — 128×128")
axes[1].axis("off")

axes[2].imshow(
    synthetic_lr_test,
    cmap="gray"
)
axes[2].set_title(
    "Synthetic NoisyLR — 128×128"
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def validate_final_generator(
    pairs,
    max_samples=1000
):

    records = []

    for gt_path, lr_path in pairs[:max_samples]:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        real_lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        synthetic_lr, info = (
            synthetic_degrade_final(gt)
        )

        records.append({

            "real_mean":
                real_lr.mean(),

            "synthetic_mean":
                synthetic_lr.mean(),

            "real_std":
                real_lr.std(),

            "synthetic_std":
                synthetic_lr.std(),

            "real_below_zero":
                np.mean(real_lr < 0),

            "synthetic_below_zero":
                np.mean(synthetic_lr < 0),

            "real_above_one":
                np.mean(real_lr > 1),

            "synthetic_above_one":
                np.mean(synthetic_lr > 1),

            "sigma_g":
                info["sigma_g"],

            "sigma_s":
                info["sigma_s"],

            "target_std":
                info["target_std"]
        })

    return pd.DataFrame(records)

In [ ]:
final_validation = validate_final_generator(
    train_pairs,
    max_samples=1000
)

display(
    final_validation.mean().to_frame(
        "Mean"
    )
)

In [ ]:
print("FINAL SYNTHETIC SEVERITY")
print("=" * 50)

for p in [5, 25, 50, 75, 95]:

    value = np.percentile(
        final_validation["target_std"],
        p
    )

    print(
        f"P{p:02d}: {value:.6f}"
    )

In [ ]:
summary = pd.DataFrame({

    "Metric": [
        "Mean intensity",
        "Std intensity",
        "Pixels < 0",
        "Pixels > 1"
    ],

    "Real": [
        final_validation["real_mean"].mean(),
        final_validation["real_std"].mean(),
        final_validation["real_below_zero"].mean(),
        final_validation["real_above_one"].mean()
    ],

    "Synthetic": [
        final_validation["synthetic_mean"].mean(),
        final_validation["synthetic_std"].mean(),
        final_validation["synthetic_below_zero"].mean(),
        final_validation["synthetic_above_one"].mean()
    ]
})

display(summary)

In [ ]:
# ============================================================
# DATA-CALIBRATED ORDER PROBABILITIES
# ============================================================

order_scores = order_results_df[
    ["Order", "Mismatch Score"]
].copy()

# Lower mismatch = higher probability
EPS = 1e-6

order_scores["weight"] = (
    1.0 /
    (order_scores["Mismatch Score"] + EPS)
)

order_scores["probability"] = (
    order_scores["weight"] /
    order_scores["weight"].sum()
)

order_scores = order_scores.sort_values(
    "probability",
    ascending=False
).reset_index(drop=True)

display(
    order_scores[
        ["Order", "Mismatch Score", "probability"]
    ]
)

In [ ]:
# ============================================================
# FINAL ORDER SAMPLER
# ============================================================

FINAL_ORDERS = [
    tuple(order.split(" → "))
    for order in order_scores["Order"]
]

FINAL_ORDER_PROBS = (
    order_scores["probability"]
    .to_numpy(dtype=np.float64)
)

print("Final degradation order probabilities")
print("=" * 60)

for order, prob in zip(
    FINAL_ORDERS,
    FINAL_ORDER_PROBS
):
    print(
        f"{' → '.join(order):45s} "
        f"{prob:.4f} "
        f"({prob * 100:.2f}%)"
    )

print(
    "\nProbability sum:",
    FINAL_ORDER_PROBS.sum()
)

In [ ]:
# ============================================================
# TEST ORDER SAMPLING
# ============================================================

from collections import Counter

N_TEST = 10000

sampled_orders = random.choices(
    FINAL_ORDERS,
    weights=FINAL_ORDER_PROBS,
    k=N_TEST
)

counts = Counter(
    sampled_orders
)

print("Observed sampling frequencies")
print("=" * 60)

for order, expected_prob in zip(
    FINAL_ORDERS,
    FINAL_ORDER_PROBS
):

    observed_prob = (
        counts[order] /
        N_TEST
    )

    print(
        f"{' → '.join(order):45s} "
        f"expected={expected_prob:.4f} "
        f"observed={observed_prob:.4f}"
    )

In [ ]:
# ============================================================
# FINAL SYNTHETIC DEGRADATION GENERATOR
# ============================================================

def synthetic_degrade_final(
    gt,
    return_info=False
):
    """
    Final synthetic degradation pipeline.

    Degradations:
        1. Gaussian noise
        2. 2x downsampling
        3. Speckle noise

    Both noise severity and operation order
    are sampled from empirical calibration.

    Returns:
        synthetic LR image
        optionally degradation metadata
    """

    # --------------------------------------------------------
    # Sample degradation severity
    # --------------------------------------------------------

    sigma_g, sigma_s, target_std, scale = (
        sample_degradation_strength()
    )

    # --------------------------------------------------------
    # Sample degradation order
    # --------------------------------------------------------

    order = random.choices(
        FINAL_ORDERS,
        weights=FINAL_ORDER_PROBS,
        k=1
    )[0]

    # --------------------------------------------------------
    # Apply degradation
    # --------------------------------------------------------

    synthetic_lr = synthetic_degrade_ordered(
        gt=gt,
        order=order,
        sigma_g=sigma_g,
        sigma_s=sigma_s
    )

    synthetic_lr = synthetic_lr.astype(
        np.float32
    )

    if return_info:

        info = {
            "order": order,
            "sigma_g": sigma_g,
            "sigma_s": sigma_s,
            "target_std": target_std,
            "severity_scale": scale
        }

        return synthetic_lr, info

    return synthetic_lr

In [ ]:
# ============================================================
# FINAL GENERATOR SANITY CHECK
# ============================================================

gt_test = np.load(
    train_pairs[0][0]
).astype(np.float32)

real_lr_test = np.load(
    train_pairs[0][1]
).astype(np.float32)

synthetic_lr_test, info = (
    synthetic_degrade_final(
        gt_test,
        return_info=True
    )
)

print("Final degradation")
print("=" * 50)

print(
    "Order       :",
    " → ".join(info["order"])
)

print(
    f"Gaussian σ  : {info['sigma_g']:.6f}"
)

print(
    f"Speckle σ   : {info['sigma_s']:.6f}"
)

print(
    f"Target std  : {info['target_std']:.6f}"
)

print(
    f"Scale       : {info['severity_scale']:.6f}"
)

print("\nShapes")
print("GT       :", gt_test.shape)
print("Real LR  :", real_lr_test.shape)
print("Synth LR :", synthetic_lr_test.shape)

In [ ]:
# ============================================================
# RANDOMIZED DEGRADATION VISUALIZATION
# ============================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 8)
)

axes[0, 0].imshow(
    gt_test,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[0, 0].set_title(
    "GT"
)

axes[0, 0].axis("off")


for i in range(1, 8):

    synthetic, info = (
        synthetic_degrade_final(
            gt_test,
            return_info=True
        )
    )

    ax = axes.flatten()[i]

    ax.imshow(
        synthetic,
        cmap="gray"
    )

    ax.set_title(
        f"{' → '.join(info['order'])}\n"
        f"σG={info['sigma_g']:.3f}, "
        f"σS={info['sigma_s']:.3f}"
    )

    ax.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VALIDATE FINAL RANDOMIZED GENERATOR
# ============================================================

def validate_randomized_generator(
    pairs,
    max_samples=1000
):

    records = []

    for gt_path, lr_path in pairs[:max_samples]:

        gt = np.load(
            gt_path,
            mmap_mode="r"
        ).astype(np.float32)

        real_lr = np.load(
            lr_path,
            mmap_mode="r"
        ).astype(np.float32)

        synthetic_lr, info = (
            synthetic_degrade_final(
                gt,
                return_info=True
            )
        )

        records.append({

            "real_mean":
                real_lr.mean(),

            "synthetic_mean":
                synthetic_lr.mean(),

            "real_std":
                real_lr.std(),

            "synthetic_std":
                synthetic_lr.std(),

            "real_below_zero":
                np.mean(real_lr < 0),

            "synthetic_below_zero":
                np.mean(synthetic_lr < 0),

            "real_above_one":
                np.mean(real_lr > 1),

            "synthetic_above_one":
                np.mean(synthetic_lr > 1),

            "target_std":
                info["target_std"],

            "order":
                " → ".join(info["order"])
        })

    return pd.DataFrame(records)

In [ ]:
final_randomized_validation = (
    validate_randomized_generator(
        train_pairs,
        max_samples=1000
    )
)

summary = pd.DataFrame({

    "Metric": [
        "Mean intensity",
        "Std intensity",
        "Pixels < 0",
        "Pixels > 1"
    ],

    "Real": [
        final_randomized_validation[
            "real_mean"
        ].mean(),

        final_randomized_validation[
            "real_std"
        ].mean(),

        final_randomized_validation[
            "real_below_zero"
        ].mean(),

        final_randomized_validation[
            "real_above_one"
        ].mean()
    ],

    "Synthetic": [
        final_randomized_validation[
            "synthetic_mean"
        ].mean(),

        final_randomized_validation[
            "synthetic_std"
        ].mean(),

        final_randomized_validation[
            "synthetic_below_zero"
        ].mean(),

        final_randomized_validation[
            "synthetic_above_one"
        ].mean()
    ]
})

display(summary)

In [ ]:
print("\nObserved final order frequencies")
print("=" * 60)

print(
    final_randomized_validation[
        "order"
    ].value_counts(normalize=True)
)

In [ ]:
# ============================================================
# NAFNET STAGE
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", torch.cuda.get_device_name(0)
      if torch.cuda.is_available()
      else "CPU")

In [ ]:
print("Training pairs :", len(train_pairs))
print("Validation pairs:", len(val_pairs))

print("\nExample:")
print("GT :", train_pairs[0][0])
print("LR :", train_pairs[0][1])

In [ ]:
# ============================================================
# FAST NAFNET TRAIN DATASET
# ============================================================

class NAFNetTrainDataset(Dataset):

    def __init__(
        self,
        pairs,
        use_synthetic=True,
        preload=True
    ):

        self.pairs = pairs
        self.use_synthetic = use_synthetic
        self.preload = preload

        if preload:

            print("Preloading GT images into RAM...")

            self.gt_cache = [
                np.load(
                    gt_path
                ).astype(np.float32)
                for gt_path, _ in pairs
            ]

            print(
                f"Loaded {len(self.gt_cache)} GT images"
            )

        else:

            self.gt_cache = None

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        # ----------------------------------------------------
        # GT
        # ----------------------------------------------------

        if self.gt_cache is not None:

            gt = self.gt_cache[idx]

        else:

            gt = np.load(
                self.pairs[idx][0]
            ).astype(np.float32)

        # ----------------------------------------------------
        # Synthetic degradation
        # ----------------------------------------------------

        if self.use_synthetic:

            lr = synthetic_degrade_final(
                gt
            )

        else:

            lr = np.load(
                self.pairs[idx][1]
            ).astype(np.float32)

        # ----------------------------------------------------
        # Torch tensors
        # ----------------------------------------------------

        lr = torch.from_numpy(
            np.ascontiguousarray(lr)
        ).unsqueeze(0)

        gt = torch.from_numpy(
            np.ascontiguousarray(gt)
        ).unsqueeze(0)

        return lr, gt

In [ ]:
class NAFNetValidationDataset(Dataset):

    def __init__(self, pairs):

        self.pairs = pairs

    def __len__(self):

        return len(self.pairs)

    def __getitem__(self, idx):

        gt_path, lr_path = self.pairs[idx]

        gt = np.load(
            gt_path
        ).astype(np.float32)

        lr = np.load(
            lr_path
        ).astype(np.float32)

        gt = torch.from_numpy(
            gt
        ).unsqueeze(0)

        lr = torch.from_numpy(
            lr
        ).unsqueeze(0)

        return lr, gt

In [ ]:
# ============================================================
# RECREATE DATASETS
# ============================================================

train_dataset = NAFNetTrainDataset(
    train_pairs,
    use_synthetic=True,
    preload=True
)

val_dataset = NAFNetValidationDataset(
    val_pairs
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Val samples:",
    len(val_dataset)
)

In [ ]:
# ============================================================
# VISUALIZE ONE IMAGE FROM THE BATCH
# Works with BOTH NumPy arrays and PyTorch tensors
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


def to_numpy_image(x):

    # --------------------------------------------------------
    # PyTorch tensor
    # --------------------------------------------------------
    if hasattr(x, "detach"):

        x = (
            x.detach()
            .cpu()
            .numpy()
        )

    # --------------------------------------------------------
    # NumPy array
    # --------------------------------------------------------
    else:

        x = np.asarray(x)


    # --------------------------------------------------------
    # Remove batch dimension
    # --------------------------------------------------------

    if x.ndim == 4:
        # [B, C, H, W]
        x = x[0]

    # --------------------------------------------------------
    # Remove channel dimension
    # --------------------------------------------------------

    if x.ndim == 3:

        # [C, H, W]
        if x.shape[0] == 1:
            x = x[0]

        # [H, W, C]
        elif x.shape[-1] == 1:
            x = x[..., 0]

    return x


# ============================================================
# CONVERT
# ============================================================

lr_vis = to_numpy_image(lr)
gt_vis = to_numpy_image(gt)


print(
    "LR visual shape:",
    lr_vis.shape
)

print(
    "GT visual shape:",
    gt_vis.shape
)

print(
    "LR dtype:",
    lr_vis.dtype
)

print(
    "GT dtype:",
    gt_vis.dtype
)


# ============================================================
# VISUALIZE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)


axes[0].imshow(
    lr_vis,
    cmap="gray"
)

axes[0].set_title(
    "Synthetic LR — 128×128"
)

axes[0].axis("off")


axes[1].imshow(
    gt_vis,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[1].set_title(
    "GT — 256×256"
)

axes[1].axis("off")


plt.tight_layout()
plt.show()

NAFNet architecture

In [ ]:
class LayerNorm2d(nn.Module):

    def __init__(
        self,
        channels,
        eps=1e-6
    ):

        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(
                channels
            )
        )

        self.bias = nn.Parameter(
            torch.zeros(
                channels
            )
        )

        self.eps = eps

    def forward(self, x):

        mean = x.mean(
            dim=1,
            keepdim=True
        )

        var = (
            x - mean
        ).pow(2).mean(
            dim=1,
            keepdim=True
        )

        x = (
            x - mean
        ) / torch.sqrt(
            var + self.eps
        )

        return (
            self.weight.view(
                1, -1, 1, 1
            ) * x
            +
            self.bias.view(
                1, -1, 1, 1
            )
        )

In [ ]:
class SimpleGate(nn.Module):

    def forward(self, x):

        x1, x2 = x.chunk(
            2,
            dim=1
        )

        return x1 * x2

In [ ]:
class NAFBlock(nn.Module):

    def __init__(
        self,
        channels,
        dw_expand=2,
        ffn_expand=2,
        dropout=0.0
    ):

        super().__init__()

        dw_channels = (
            channels * dw_expand
        )

        ffn_channels = (
            channels * ffn_expand
        )

        # ----------------------------------------------------
        # Spatial branch
        # ----------------------------------------------------

        self.norm1 = LayerNorm2d(
            channels
        )

        self.conv1 = nn.Conv2d(
            channels,
            dw_channels,
            kernel_size=1,
            bias=True
        )

        self.dwconv = nn.Conv2d(
            dw_channels,
            dw_channels,
            kernel_size=3,
            padding=1,
            groups=dw_channels,
            bias=True
        )

        self.simple_gate = SimpleGate()

        # Channel attention
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=1,
                bias=True
            )
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            bias=True
        )

        self.dropout1 = nn.Dropout(
            dropout
        )

        self.beta = nn.Parameter(
            torch.zeros(
                (1, channels, 1, 1)
            )
        )

        # ----------------------------------------------------
        # Feed-forward branch
        # ----------------------------------------------------

        self.norm2 = LayerNorm2d(
            channels
        )

        self.conv3 = nn.Conv2d(
            channels,
            ffn_channels * 2,
            kernel_size=1,
            bias=True
        )

        self.simple_gate2 = SimpleGate()

        self.conv4 = nn.Conv2d(
            ffn_channels,
            channels,
            kernel_size=1,
            bias=True
        )

        self.dropout2 = nn.Dropout(
            dropout
        )

        self.gamma = nn.Parameter(
            torch.zeros(
                (1, channels, 1, 1)
            )
        )

    def forward(self, x):

        # ----------------------------------------------------
        # Spatial branch
        # ----------------------------------------------------

        y = self.norm1(x)

        y = self.conv1(y)
        y = self.dwconv(y)
        y = self.simple_gate(y)

        y = y * self.sca(y)

        y = self.conv2(y)

        y = self.dropout1(y)

        x = x + self.beta * y

        # ----------------------------------------------------
        # Feed-forward branch
        # ----------------------------------------------------

        y = self.norm2(x)

        y = self.conv3(y)
        y = self.simple_gate2(y)
        y = self.conv4(y)

        y = self.dropout2(y)

        x = x + self.gamma * y

        return x

In [ ]:
class NAFNetSR(nn.Module):

    def __init__(
        self,
        img_channel=1,
        width=32,
        enc_blocks=(2, 2, 4),
        middle_blocks=4,
        dec_blocks=(2, 2, 2)
    ):

        super().__init__()

        self.intro = nn.Conv2d(
            img_channel,
            width,
            kernel_size=3,
            padding=1
        )

        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()

        channels = width

        for num_blocks in enc_blocks:

            self.encoders.append(
                nn.Sequential(
                    *[
                        NAFBlock(channels)
                        for _ in range(num_blocks)
                    ]
                )
            )

            self.downs.append(
                nn.Conv2d(
                    channels,
                    channels * 2,
                    kernel_size=2,
                    stride=2
                )
            )

            channels *= 2

        self.middle = nn.Sequential(
            *[
                NAFBlock(channels)
                for _ in range(middle_blocks)
            ]
        )

        self.ups = nn.ModuleList()
        self.decoders = nn.ModuleList()

        for num_blocks in dec_blocks:

            self.ups.append(
                nn.Sequential(
                    nn.Conv2d(
                        channels,
                        channels * 2,
                        kernel_size=1
                    ),
                    nn.PixelShuffle(2)
                )
            )

            channels //= 2

            self.decoders.append(
                nn.Sequential(
                    *[
                        NAFBlock(channels)
                        for _ in range(num_blocks)
                    ]
                )
            )

        self.up2 = nn.Sequential(
            nn.Conv2d(
                width,
                width * 4,
                kernel_size=3,
                padding=1
            ),
            nn.PixelShuffle(2)
        )

        self.outro = nn.Conv2d(
            width,
            img_channel,
            kernel_size=3,
            padding=1
        )

    def forward(self, x):

        # Keep bicubic input as global reconstruction baseline
        base = F.interpolate(
            x,
            scale_factor=2,
            mode="bicubic",
            align_corners=False
        )

        x = self.intro(x)

        skips = []

        for encoder, down in zip(
            self.encoders,
            self.downs
        ):

            x = encoder(x)

            skips.append(x)

            x = down(x)

        x = self.middle(x)

        for up, decoder, skip in zip(
            self.ups,
            self.decoders,
            reversed(skips)
        ):

            x = up(x)

            x = x + skip

            x = decoder(x)

        x = self.up2(x)

        residual = self.outro(x)

        return base + residual

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = NAFNetSR(
    img_channel=1,
    width=32,
    enc_blocks=(2, 2, 4),
    middle_blocks=4,
    dec_blocks=(2, 2, 2)
).to(device)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

In [ ]:
# ============================================================
# NAFNET SHAPE TEST
# ============================================================

import numpy as np
import torch

# ------------------------------------------------------------
# Convert LR to PyTorch tensor if it is currently NumPy
# ------------------------------------------------------------

if isinstance(lr, np.ndarray):
    test_input = torch.from_numpy(lr).float()
else:
    test_input = lr.float()


# ------------------------------------------------------------
# Make sure batch dimension exists
# ------------------------------------------------------------

if test_input.ndim == 2:
    # [H, W]
    test_input = test_input.unsqueeze(0).unsqueeze(0)

elif test_input.ndim == 3:
    # [B, H, W] OR [C, H, W]
    #
    # In this notebook LR is grayscale.
    if test_input.shape[0] == 1:
        test_input = test_input.unsqueeze(0)
    else:
        test_input = test_input.unsqueeze(1)


# ------------------------------------------------------------
# Move to GPU
# ------------------------------------------------------------

test_input = test_input.to(device)


# ------------------------------------------------------------
# NAFNet inference
# ------------------------------------------------------------

model = model.to(device)
model.eval()

with torch.no_grad():

    test_output = model(
        test_input
    )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 60)
print("NAFNET SHAPE TEST")
print("=" * 60)

print(
    "Input :",
    tuple(test_input.shape)
)

print(
    "Output:",
    tuple(test_output.shape)
)

print(
    "Input dtype :",
    test_input.dtype
)

print(
    "Output dtype:",
    test_output.dtype
)

print("=" * 60)

In [ ]:
# ============================================================
# FAST DATALOADERS
# ============================================================

BATCH_SIZE = 8
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)

In [ ]:
# ============================================================
# BATCH SANITY CHECK
# ============================================================

lr_batch, gt_batch = next(
    iter(train_loader)
)

print("LR batch")
print("shape :", lr_batch.shape)
print("dtype :", lr_batch.dtype)
print("min   :", lr_batch.min().item())
print("max   :", lr_batch.max().item())

print("\nGT batch")
print("shape :", gt_batch.shape)
print("dtype :", gt_batch.dtype)
print("min   :", gt_batch.min().item())
print("max   :", gt_batch.max().item())

In [ ]:
# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = nn.L1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

print("Initial LR:", optimizer.param_groups[0]["lr"])

In [ ]:
# ============================================================
# METRICS
# ============================================================

def calculate_psnr(
    prediction,
    target,
    data_range=1.0
):

    prediction = prediction.detach()
    target = target.detach()

    mse = F.mse_loss(
        prediction,
        target
    )

    if mse.item() == 0:
        return float("inf")

    return (
        10.0 *
        torch.log10(
            torch.tensor(
                data_range ** 2,
                device=mse.device
            ) / mse
        )
    ).item()

In [ ]:
from skimage.metrics import structural_similarity

In [ ]:
def calculate_ssim(
    prediction,
    target
):

    prediction = (
        prediction.detach()
        .cpu()
        .numpy()
    )

    target = (
        target.detach()
        .cpu()
        .numpy()
    )

    scores = []

    for p, t in zip(
        prediction,
        target
    ):

        p = p.squeeze()
        t = t.squeeze()

        scores.append(
            structural_similarity(
                t,
                p,
                data_range=1.0
            )
        )

    return float(
        np.mean(scores)
    )

In [ ]:
def clip_for_metrics(x):

    return torch.clamp(
        x,
        0.0,
        1.0
    )

In [ ]:
# ============================================================
# FASTER REAL-PAIR VALIDATION
# ============================================================

@torch.no_grad()
def validate_model(
    model,
    loader,
    calculate_ssim_metric=True,
    max_ssim_samples=None
):

    model.eval()

    total_loss = 0.0
    total_psnr = 0.0

    total_ssim = 0.0
    ssim_count = 0

    count = 0

    for lr, gt in loader:

        lr = lr.to(
            device,
            non_blocking=True
        )

        gt = gt.to(
            device,
            non_blocking=True
        )

        prediction = model(lr)

        loss = criterion(
            prediction,
            gt
        )

        metric_prediction = torch.clamp(
            prediction,
            0.0,
            1.0
        )

        batch_size = lr.size(0)

        total_loss += (
            loss.item() *
            batch_size
        )

        total_psnr += (
            calculate_psnr(
                metric_prediction,
                gt
            ) *
            batch_size
        )

        count += batch_size

        # ----------------------------------------------------
        # SSIM only when requested
        # ----------------------------------------------------

        if calculate_ssim_metric:

            if (
                max_ssim_samples is None
                or ssim_count < max_ssim_samples
            ):

                current_batch = min(
                    batch_size,
                    max_ssim_samples - ssim_count
                    if max_ssim_samples is not None
                    else batch_size
                )

                ssim_value = calculate_ssim(
                    metric_prediction[
                        :current_batch
                    ],
                    gt[
                        :current_batch
                    ]
                )

                total_ssim += (
                    ssim_value *
                    current_batch
                )

                ssim_count += current_batch

    result = {
        "loss": total_loss / count,
        "psnr": total_psnr / count
    }

    if calculate_ssim_metric:

        result["ssim"] = (
            total_ssim / ssim_count
        )

    else:

        result["ssim"] = None

    return result

In [ ]:
# ============================================================
# VALIDATION SANITY CHECK
# ============================================================

val_result = validate_model(
    model,
    val_loader
)

print("Initial validation")
print("=" * 40)

print(
    f"L1   : {val_result['loss']:.6f}"
)

print(
    f"PSNR : {val_result['psnr']:.4f} dB"
)

print(
    f"SSIM : {val_result['ssim']:.6f}"
)

In [ ]:
# ============================================================
# MIXED PRECISION
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print(
    "AMP enabled:",
    torch.cuda.is_available()
)

In [ ]:
# ============================================================
# DATA PIPELINE SPEED TEST
# ============================================================

import time

loader_iter = iter(train_loader)

start = time.time()

for i in range(20):

    lr_batch, gt_batch = next(
        loader_iter
    )

elapsed = time.time() - start

print(
    f"20 batches loaded in {elapsed:.2f} sec"
)

print(
    f"Average batch loading time: "
    f"{elapsed / 20:.3f} sec"
)

print(
    "Batch shape:",
    lr_batch.shape,
    gt_batch.shape
)

In [ ]:
# ============================================================
# NAFNET TRAINING — L1 + HIGH-FREQUENCY LOSS
# ============================================================

import time
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

NUM_EPOCHS = 20
LR = 2e-4
HF_WEIGHT = 0.05

HF_CHECKPOINT_PATH = (
    "/kaggle/working/nafnet_hf_best.pth"
)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

train_losses = []
val_losses = []
val_psnrs = []
val_ssims = []

best_psnr = -float("inf")
best_ssim = -float("inf")
best_epoch = 0


# ============================================================
# HIGH-FREQUENCY MAP
# ============================================================

def high_frequency_map(x):

    kernel = torch.tensor(
        [[1., 2., 1.],
         [2., 4., 2.],
         [1., 2., 1.]],
        device=x.device,
        dtype=x.dtype
    ) / 16.0

    kernel = kernel.view(
        1, 1, 3, 3
    )

    smooth = F.conv2d(
        x,
        kernel,
        padding=1
    )

    return x - smooth


# ============================================================
# HIGH-FREQUENCY LOSS
# ============================================================

def high_frequency_loss(
    pred,
    target
):

    pred_hf = high_frequency_map(
        pred
    )

    target_hf = high_frequency_map(
        target
    )

    return F.l1_loss(
        pred_hf,
        target_hf
    )


# ============================================================
# TRAINING
# ============================================================

print("=" * 70)
print("NAFNET + HIGH-FREQUENCY LOSS")
print("=" * 70)

print(
    f"Epochs       : {NUM_EPOCHS}"
)

print(
    f"Learning rate: {LR:.2e}"
)

print(
    f"HF weight    : {HF_WEIGHT}"
)

print(
    f"Checkpoint   : {HF_CHECKPOINT_PATH}"
)

print("=" * 70)


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    start_time = time.time()

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    running_loss = 0.0
    running_l1 = 0.0
    running_hf = 0.0

    num_train_samples = 0

    for lr_batch, gt_batch in train_loader:

        lr_batch = lr_batch.to(
            device,
            non_blocking=True
        )

        gt_batch = gt_batch.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # Forward
        output = model(
            lr_batch
        )

        # Keep output in valid image range
        output = torch.clamp(
            output,
            0.0,
            1.0
        )

        # ----------------------------------------------------
        # L1 reconstruction loss
        # ----------------------------------------------------

        l1_loss = F.l1_loss(
            output,
            gt_batch
        )

        # ----------------------------------------------------
        # High-frequency preservation loss
        # ----------------------------------------------------

        hf_loss = high_frequency_loss(
            output,
            gt_batch
        )

        # ----------------------------------------------------
        # Combined objective
        # ----------------------------------------------------

        loss = (
            l1_loss
            + HF_WEIGHT * hf_loss
        )

        # Backpropagation
        loss.backward()

        optimizer.step()

        batch_size = (
            lr_batch.shape[0]
        )

        running_loss += (
            loss.item()
            * batch_size
        )

        running_l1 += (
            l1_loss.item()
            * batch_size
        )

        running_hf += (
            hf_loss.item()
            * batch_size
        )

        num_train_samples += batch_size

    # Average training losses
    epoch_train_loss = (
        running_loss
        / num_train_samples
    )

    epoch_train_l1 = (
        running_l1
        / num_train_samples
    )

    epoch_train_hf = (
        running_hf
        / num_train_samples
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    running_val_l1 = 0.0
    num_val_samples = 0

    all_psnr = []
    all_ssim = []

    with torch.no_grad():

        for lr_batch, gt_batch in val_loader:

            lr_batch = lr_batch.to(
                device,
                non_blocking=True
            )

            gt_batch = gt_batch.to(
                device,
                non_blocking=True
            )

            output = model(
                lr_batch
            )

            output = torch.clamp(
                output,
                0.0,
                1.0
            )

            # Validation remains pure L1.
            # We do NOT include HF loss here.
            val_l1 = F.l1_loss(
                output,
                gt_batch
            )

            batch_size = (
                lr_batch.shape[0]
            )

            running_val_l1 += (
                val_l1.item()
                * batch_size
            )

            num_val_samples += (
                batch_size
            )

            # ------------------------------------------------
            # PSNR
            # ------------------------------------------------

            mse = torch.mean(
                (
                    output
                    - gt_batch
                ) ** 2,
                dim=(1, 2, 3)
            )

            psnr = (
                10.0
                * torch.log10(
                    1.0 / (
                        mse + 1e-10
                    )
                )
            )

            all_psnr.extend(
                psnr.detach()
                .cpu()
                .tolist()
            )

            # ------------------------------------------------
            # SSIM
            # ------------------------------------------------

            for i in range(
                batch_size
            ):

                pred_i = output[
                    i:i+1
                ]

                gt_i = gt_batch[
                    i:i+1
                ]

                try:

                    ssim_value = calculate_ssim(
                        pred_i,
                        gt_i
                    )

                    if torch.is_tensor(
                        ssim_value
                    ):
                        ssim_value = (
                            ssim_value
                            .item()
                        )

                    all_ssim.append(
                        float(ssim_value)
                    )

                except Exception:

                    # If the existing SSIM function
                    # has a different expected format,
                    # skip this sample rather than
                    # crashing the whole training run.

                    pass

    epoch_val_l1 = (
        running_val_l1
        / num_val_samples
    )

    epoch_psnr = sum(
        all_psnr
    ) / len(all_psnr)

    if len(all_ssim) > 0:

        epoch_ssim = (
            sum(all_ssim)
            / len(all_ssim)
        )

    else:

        epoch_ssim = float("nan")

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    train_losses.append(
        epoch_train_loss
    )

    val_losses.append(
        epoch_val_l1
    )

    val_psnrs.append(
        epoch_psnr
    )

    val_ssims.append(
        epoch_ssim
    )

    # --------------------------------------------------------
    # Best checkpoint
    #
    # We use PSNR as the primary selection metric,
    # exactly as before.
    # --------------------------------------------------------

    is_best = (
        epoch_psnr > best_psnr
    )

    if is_best:

        best_psnr = epoch_psnr
        best_ssim = epoch_ssim
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "best_psnr":
                    best_psnr,

                "best_ssim":
                    best_ssim,

                "hf_weight":
                    HF_WEIGHT,

                "train_l1":
                    epoch_train_l1,

                "train_hf":
                    epoch_train_hf,

                "train_loss":
                    epoch_train_loss,

                "val_l1":
                    epoch_val_l1,
            },
            HF_CHECKPOINT_PATH
        )

    elapsed = (
        time.time()
        - start_time
    )

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Time {elapsed:.1f}s | "
        f"Train L1 {epoch_train_l1:.5f} | "
        f"Train HF {epoch_train_hf:.5f} | "
        f"Train Total {epoch_train_loss:.5f} | "
        f"Val L1 {epoch_val_l1:.5f} | "
        f"PSNR {epoch_psnr:.3f} dB | "
        f"SSIM {epoch_ssim:.4f} | "
        f"LR {optimizer.param_groups[0]['lr']:.2e}"
        + (
            " ★ BEST"
            if is_best
            else ""
        )
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("HF-LOSS EXPERIMENT COMPLETE")
print("=" * 70)

print(
    f"Best epoch : {best_epoch}"
)

print(
    f"Best PSNR  : {best_psnr:.4f} dB"
)

print(
    f"Best SSIM  : {best_ssim:.6f}"
)

print(
    f"HF weight  : {HF_WEIGHT}"
)

print(
    f"Checkpoint : {HF_CHECKPOINT_PATH}"
)

print("=" * 70)

## Save the final reproducible checkpoint

The best checkpoint selected during training is copied into a clearly named final artifact.

The checkpoint contains:
- model state
- optimizer state
- best validation PSNR/SSIM
- HF-loss weight
- training hyperparameters


In [ ]:
# ============================================================
# SAVE FINAL SUBMITTED CHECKPOINT
# ============================================================

import os
import shutil
import torch

FINAL_DIR = "/kaggle/working/final_model"
os.makedirs(FINAL_DIR, exist_ok=True)

# Best checkpoint produced by the training loop
BEST_CHECKPOINT = HF_CHECKPOINT_PATH

# Final repository-friendly name
FINAL_CHECKPOINT = os.path.join(
    FINAL_DIR,
    "nafnet_hf_final_checkpoint.pth"
)

if not os.path.exists(BEST_CHECKPOINT):
    raise FileNotFoundError(
        f"Best checkpoint not found: {BEST_CHECKPOINT}"
    )

shutil.copy2(
    BEST_CHECKPOINT,
    FINAL_CHECKPOINT
)

print("=" * 70)
print("FINAL NAFNET + HF-LOSS CHECKPOINT SAVED")
print("=" * 70)
print("Path       :", FINAL_CHECKPOINT)
print("Size (MB)  :", os.path.getsize(FINAL_CHECKPOINT) / (1024 ** 2))
print("Best epoch :", best_epoch)
print("Best PSNR  :", best_psnr)
print("Best SSIM  :", best_ssim)
print("HF weight  :", HF_WEIGHT)
print("=" * 70)


In [ ]:
# ============================================================
# VERIFY FINAL CHECKPOINT
# ============================================================

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu"
)

# Recreate the exact submitted architecture
verify_model = NAFNetSR(
    img_channel=1,
    width=32,
    enc_blocks=(2, 2, 4),
    middle_blocks=4,
    dec_blocks=(2, 2, 2)
).cpu()

result = verify_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print("=" * 70)
print("FINAL CHECKPOINT VERIFICATION")
print("=" * 70)
print("Checkpoint loaded      : PASS")
print("Strict state_dict load : PASS")
print("Missing keys           :", result.missing_keys)
print("Unexpected keys        :", result.unexpected_keys)
print("Best epoch             :", checkpoint.get("epoch"))
print("Best PSNR              :", checkpoint.get("best_psnr"))
print("Best SSIM              :", checkpoint.get("best_ssim"))
print("HF weight              :", checkpoint.get("hf_weight"))
print("Parameters             :", sum(p.numel() for p in verify_model.parameters()))
print("=" * 70)

assert result.missing_keys == []
assert result.unexpected_keys == []

# Forward shape verification
x = torch.rand(1, 1, 128, 128)

with torch.no_grad():
    y = verify_model(x)

print("Input shape            :", tuple(x.shape))
print("Output shape           :", tuple(y.shape))

assert tuple(y.shape) == (1, 1, 256, 256)

print("Forward shape test     : PASS")
print("=" * 70)
